In [1]:
import os
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [2]:
# --- CONFIGURATION ---
INPUT_DIR = "processed_data/"
MODEL_DIR = "saved_models/"
SUBMISSION_FILE = "submission.csv"
ENSEMBLE_SIZE = 3           
ESM_DIM = 1280

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# --- MODEL ARCHITECTURE (from Phase 3) ---
class DTA_Model(torch.nn.Module):
    def __init__(self, node_features=29, esm_dim=ESM_DIM, hidden_dim=256):
        super(DTA_Model, self).__init__()
        
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim * 2)
        self.conv3 = GCNConv(hidden_dim * 2, hidden_dim)
        
        self.prot_proj = torch.nn.Sequential(
            torch.nn.Linear(esm_dim, hidden_dim * 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden_dim * 2, hidden_dim)
        )
        
        self.fc1 = torch.nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, int(hidden_dim/2))
        self.out = torch.nn.Linear(int(hidden_dim/2), 1)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        target_emb = data.target_emb
        
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        drug_rep = global_mean_pool(x, batch) 
        
        prot_rep = self.prot_proj(target_emb)
        cat_rep = torch.cat([drug_rep, prot_rep], dim=1)
        
        h = F.relu(self.fc1(cat_rep))
        h = self.dropout(h)
        h = F.relu(self.fc2(h))
        pred = self.out(h)
        
        return pred.squeeze(-1)

In [4]:
# --- LOAD DATA AND MODELS ---
print("Loading test data...")
test_data = torch.load(os.path.join(INPUT_DIR, "test_data.pt"))
# Shuffle=False is CRITICAL here so the IDs align perfectly with the predictions
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

print(f"Loading {ENSEMBLE_SIZE} models from disk...")
models = []
for i in range(1, ENSEMBLE_SIZE + 1):
    model_path = os.path.join(MODEL_DIR, f"dta_ensemble_model_{i}.pt")
    
    # Initialize a fresh model structure
    model = DTA_Model().to(device)
    
    # Load the trained weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    
    # Set to evaluation mode (turns off Dropout and BatchNorm behavior)
    model.eval() 
    models.append(model)

Loading test data...
Loading 3 models from disk...


In [5]:
# --- INFERENCE LOOP ---
print("\nStarting ensemble inference...")

all_ids = []
# Create a list of empty lists to store predictions for each model independently
# dim: [ENSEMBLE_SIZE, Total_Test_Samples]
ensemble_preds = [[] for _ in range(ENSEMBLE_SIZE)] 

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting Test Set"):
        batch = batch.to(device)
        
        # 1. Extract IDs for this batch (only need to do this once per batch)
        # dim: batch.sample_id is usually a list or 1D tensor [B]
        batch_ids = batch.sample_id
        
        # If it's a tensor, move to CPU and convert to list/numpy
        if torch.is_tensor(batch_ids):
            batch_ids = batch_ids.cpu().numpy()
            
        all_ids.extend(batch_ids)
        
        # 2. Get predictions from EVERY model in the ensemble for this batch
        for i, model in enumerate(models):
            preds = model(batch) # dim: [B]
            ensemble_preds[i].extend(preds.cpu().numpy())


Starting ensemble inference...


Predicting Test Set: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [00:06<00:00, 29.19it/s]


In [6]:
# --- AGGREGATE AND SAVE ---
print("\nAggregating predictions...")

# Convert list of lists to a 2D numpy array
# dim: [3, 23191] (Assuming ENSEMBLE_SIZE=3 and 23,191 test samples)
ensemble_preds_array = np.array(ensemble_preds) 

# Average across the models (axis=0 means average down the columns)
# dim: [23191]
final_avg_preds = np.mean(ensemble_preds_array, axis=0)

# Optional: Calculate disagreement (variance) between models. 
# High variance = High Epistemic Uncertainty. Good for Phase 4 / OOD analysis!
pred_variance = np.var(ensemble_preds_array, axis=0)

print(f"Sanity Check - Total IDs: {len(all_ids)}, Total Preds: {len(final_avg_preds)}")

# Build the final DataFrame
submission_df = pd.DataFrame({
    'id': all_ids,
    'affinity': final_avg_preds
})

# Save to CSV
submission_df.to_csv(SUBMISSION_FILE, index=False)
print(f"Success! Submission saved to {SUBMISSION_FILE}")

# Quick peek at the top of the file
print("\nSubmission Head:")
print(submission_df.head())


Aggregating predictions...
Sanity Check - Total IDs: 23191, Total Preds: 23191
Success! Submission saved to submission.csv

Submission Head:
           id  affinity
0  D_99418882  5.101646
1  D_62627379  4.975663
2  D_45421420  5.284131
3  D_36975128  4.601133
4  D_21152419  4.720267


In [9]:
pred_variance

array([0.13127221, 0.14303677, 0.12123897, ..., 0.32862958, 0.14567687,
       0.09860244], dtype=float32)